<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açınız](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-11/NB11_final_projesi_atolyesi.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-11/ders.html)

# Week 11: Final project and graduation

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lab.html) · [PDF notes](Week11_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-11/NB11_final_project_workshop.ipynb)

</div>

**How to use this Jupyter notebook.** The notebook follows the practice stops of the lecture page. Every stop has three parts, and each part also runs on its own: Python basics, application with ready prompts, and the ready application. **Python basics** teaches the Python that the stop needs, with small examples from the topic of the week; its exercises check themselves. **Prompt and application** gives ready prompts for an artificial intelligence (AI) assistant: The assistant writes the code, you run it, and a check cell tells you whether the result is right. **Ready application** is the complete working Python program of the stop, with notes that explain it.

Go through the three parts in this order. If you already know Python, you may start with the prompts; if you only want to see the result, open the ready application. The weekly project at the end stands on its own as well.

**Start cells.** Every part opens with a start cell. It prepares the course package and the packages of the week, and it repeats the data calls and the code of the earlier parts that the cells of the part use. The part therefore runs without errors even when you start there. If you have run the earlier parts in this session, you do not need to run the start cell; running it again does no harm. A few sections of the ready applications, which the lecture page opens directly, have a start cell of their own.

**Data.** The Cleveland data are read from the course folder, or from the UCI Machine Learning Repository when the file is not there; nothing else is downloaded this week.

**Finding your place.** The table of contents lists every stop and part. In Colab it opens with the list icon on the left; in Jupyter, with View → Table of Contents. Every part ends with a link back to the lecture page. In Colab, your changes are kept only in a copy saved to Drive: Use File → Save a copy in Drive.

**Where it runs.** The notebook runs in Jupyter (Anaconda) on your own computer and in Google Colab. Every data set here is synthetic or comes from a public research database. Real patient data belong only on a computer of your institution, under its rules, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells hold Python code. Click into the code cell below and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers everything that the cells have run, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

**The start cell.** The code cell below prepares the session. It finds the course package `vclinic` in the course folder; if the notebook was opened on its own, for example in Google Colab, it downloads the package from GitHub once. It then loads the packages, chooses the language of the messages and prepares the expected answers of the exercises. Its last lines import the libraries of the week. Every part of the notebook opens with a start cell that holds the same setup, together with the data calls of the part.

In [ ]:
# Start cell of the Jupyter notebook
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the libraries of the week

import importlib
import json
import math
import platform

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

## Stop 1: The question, the data and a reference

*Lecture page: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#stop-1)*

This stop is the first half of the workshop on the final projects, on the worked project of the week. The data of the Cleveland patients are audited, the size that a model of ten parameters needs is compared with the training patients, and the reference of age, sex and chest pain type is tested once on the test patients. After the stop, the same three steps are repeated on the data of your own final project.

**At the end of this stop:** The missing values in the columns that the project leaves out (6), the size that criterion (i) of Riley and colleagues asks for (241 patients) and the test AUC of the reference (0.900).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 1 · Python basics

This week the code of the course becomes the code of a project of your own. Three habits make a project that others can check: a folder in which every file has its place, functions that say what they do, and results that come out the same every time the notebook runs. This part shows how `pathlib` makes the folders of a project, how a *docstring* documents a function, how a generator of random numbers started with a *seed* draws a resample that can be repeated, and how the versions of the packages are written next to the results.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 1 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
import json
import platform
import sklearn

In [ ]:
# Python practice: a small example for learning the language
project = pathlib.Path("final_project")
for name in ["data", "results"]:
    (project / name).mkdir(parents=True, exist_ok=True)
readme = project / "README.md"
if not readme.exists():
    readme.write_text("# Ruling out coronary artery disease after an exercise test\n", encoding="utf-8")
print(sorted(entry.name for entry in project.iterdir()))

**Python note.** `pathlib.Path("final_project")` is a path, not yet a folder. The operator `/` joins paths, so `project / "data"` is the subfolder data. `mkdir(parents=True, exist_ok=True)` makes the folder and any folder above it, and does nothing when it exists already, so the cell can run twice. `readme.exists()` asks whether the file is there before `write_text` writes it, so a README that you have edited is never overwritten. `project.iterdir()` lists the entries of the folder, and `sorted(...)` puts their names in order.

In [ ]:
# Python practice: a small example for learning the language
def share_with_disease(outcomes):
    """Share of the patients with the outcome 1.

    outcomes: a list of 0 and 1, one value per patient.
    Returns a number between 0 and 1."""
    return sum(outcomes) / len(outcomes)


p10 = [0.047, 0.092, 0.168, 0.198, 0.304, 0.466, 0.608, 0.691, 0.936, 0.978]
y10 = [0, 0, 0, 1, 0, 1, 0, 1, 1, 1]
print(share_with_disease(y10))
help(share_with_disease)

**Python note.** The text in triple quotes right under `def` is the *docstring* of the function: what it does, what it expects and what it returns. `help(share_with_disease)` prints it, as it prints the help of every function of NumPy or pandas. A docstring is the shortest form of documentation, and it travels with the code. `p10` and `y10` are the probabilities and the outcomes of the ten patients of the worked example of the bootstrap on the lecture page.

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(2026)
draw = rng.integers(0, 10, 10)
again = np.random.default_rng(2026).integers(0, 10, 10)
print(draw, again, bool((draw == again).all()))
print("patients of the resample, numbered from 1:", sorted(int(k) + 1 for k in draw))
print("second draw of the same generator:", rng.integers(0, 10, 10))

**Python note.** `np.random.default_rng(2026)` makes a generator of random numbers started with the seed 2026. `rng.integers(0, 10, 10)` draws ten whole numbers from 0 to 9, the upper limit 10 excluded. A second generator with the same seed draws exactly the same numbers, so a bootstrap with a seed gives the same interval on every computer. The generator moves on with every draw: Its second draw differs from its first, which is what a bootstrap of 2,000 resamples needs. The numbers drawn here are the resample of the worked example of the lecture: P1, P1, P2, P4, P4, P4, P5, P7, P7 and P9.

In [ ]:
# Python practice: a small example for learning the language
summary = {"patients": len(y10), "share with disease": share_with_disease(y10), "seed": 2026,
           "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
(project / "results" / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

**Python note.** A result is only reproducible with the conditions under which it was computed. `platform.python_version()` gives the version of Python, and `np.__version__`, `pd.__version__` and `sklearn.__version__` those of the packages. The dictionary `summary` keeps them with the result and the seed, and `json.dumps(..., indent=2)` writes it into `final_project/results/summary.json` as a text that any program can read; `ensure_ascii=False` keeps the Turkish letters as they are.

**Quick check.** Why does the file of results hold the seed of the random numbers and the versions of the packages?

<details><summary>Answer</summary>

So that another person, or you a year later, can draw the same resamples with the same code and get the same intervals. If a result differs anyway, the versions show whether a package changed in between.

</details>

**Exercise 1.1.** With a generator started with the seed 2026, draw ten whole numbers from 0 to 9, as above. How many different patients does the resample hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(set(np.random.default_rng(2026).integers(0, 10, 10).tolist()))
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Add the subfolders code and figures to `project`, next to data and results. How many entries does the folder final_project hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
for name in ["data", "code", "results", "figures"]:
    (project / name).mkdir(parents=True, exist_ok=True)
answer = len(list(project.iterdir()))
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** Write a function `auc_pairs(y, p)` with a docstring that returns the share of the pairs of a patient with and a patient without disease in which the patient with disease has the higher probability, a tie counting one half. What is the AUC of the resample `draw` of the ten patients `p10` and `y10`? Keep it with three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def auc_pairs(y, p):
    """Share of the pairs (disease, no disease) in the right order; a tie counts one half."""
    pos = [pi for yi, pi in zip(y, p) if yi == 1]
    neg = [pi for yi, pi in zip(y, p) if yi == 0]
    return sum(1.0 if a > b else 0.5 if a == b else 0.0 for a in pos for b in neg) / (len(pos) * len(neg))


answer = round(auc_pairs([y10[k] for k in draw], [p10[k] for k in draw]), 3)
teach.check("Exercise 1.3", answer)

**«Stop 1 · Python basics» is complete.** Next comes «Stop 1 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#model-training-optimisation-and-xai-integration) and continue reading at «Model training, optimisation and XAI integration».

### Stop 1 · Prompt and application

This part has three prompts: «The audit of the data», «How much data the model needs» and «The reference on the test patients».

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Working with prompts.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab and opens with the Gemini button at the top of the window; ChatGPT, Claude, Copilot or Gemini in another browser tab work as well. All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-11/PROMPTS.md), where GitHub shows a copy button next to each one.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. For each prompt, the data cell puts its data into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

**Start cell.** For «Stop 1 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _course():
    """The worked project of the week: the Cleveland data, the split of Week 2, the reference, the full model and the cross-validated probabilities of the training patients (computed once, a few seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland(source=DATA_SOURCE)
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"missing values": missing, "used by the project": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """The Cox-Snell R squared of predicted probabilities p for the outcomes y."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """The three sizes of Riley and colleagues for a model of a binary outcome with P parameters: (i) a shrinkage of at least 0.9, (ii) apparent and adjusted Nagelkerke R squared within 0.05, (iii) the overall risk within 0.05."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """The training or the test patients, or both in the order of the split, with their set, the eight clinical variables and the disease."""
    c = _course()
    parts = {"training": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"patient": h.index, "set": [s for s, _ in rows],
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "chest pain type": h["cp"].map(_CP),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "disease": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "patients of the Cleveland data:", len(_c["X_train"]), "for training and", len(_c["X_test"]), "for the test")

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.metrics import roc_auc_score, roc_curve

#### Prompt 1.1: The audit of the data

**Goal.** Count the missing values of the data, column by column, and see which columns the project uses.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A cardiologist plans a final project on ruling out coronary artery disease after an exercise test, so that some patients need not be referred for angiography. Before any model, the data are audited: Which columns have missing values, and are they columns that the decision would have at its moment?  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test, with all their columns. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302). The eight variables of the project are "age" (years), "sex" ("man" or "woman"), "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina) and "ST depression" (the depression of the ST segment during exercise compared with rest, mm). Five more columns come from the data set: "fasting blood sugar above 120" (1 when the fasting blood sugar was above 120 mg/dL), "resting ECG" (0 normal, 1 an abnormality of the ST segment or the T wave, 2 left ventricular hypertrophy), "ST slope" (the slope of the ST segment at peak exercise: 1 up, 2 flat, 3 down), "vessels on fluoroscopy" (the number of major vessels coloured by fluoroscopy, 0 to 3) and "thallium scan" (3 normal, 6 fixed defect, 7 reversible defect). An empty cell is a missing value. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Count the missing values of every column except "patient". The project uses nine columns: the eight variables and "disease". Mark for each column whether the project uses it, and add up the missing values of the columns that it does not use. Also give the number of patients, the number with disease and their share.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with the missing values of each column and whether the project uses it, print the counts of the patients, and keep the number of missing values in the columns that the project does not use under the name "answer".  
> Test: The nine columns that the project uses must have no missing value, and the table must hold 303 patients.  

In [ ]:
# Data of Prompt 1.1: the table "data" that the prompt describes
_h = _course()["heart"]
data = pd.DataFrame({"patient": _h.index, "age": _h["age"], "sex": _h["sex"].map({1.0: "man", 0.0: "woman"}),
                     "chest pain type": _h["cp"].map(_CP), "resting blood pressure": _h["trestbps"],
                     "cholesterol": _h["chol"], "fasting blood sugar above 120": _h["fbs"],
                     "resting ECG": _h["restecg"], "maximum heart rate": _h["thalach"],
                     "exercise angina": _h["exang"], "ST depression": _h["oldpeak"], "ST slope": _h["slope"],
                     "vessels on fluoroscopy": _h["ca"], "thallium scan": _h["thal"],
                     "disease": _h["disease"]}).reset_index(drop=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.1
teach.check_step("Prompt 1.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.1 (run it only if the code of the assistant does not pass the check)
_h = _course()["heart"]
data = pd.DataFrame({"patient": _h.index, "age": _h["age"], "sex": _h["sex"].map({1.0: "man", 0.0: "woman"}),
                     "chest pain type": _h["cp"].map(_CP), "resting blood pressure": _h["trestbps"],
                     "cholesterol": _h["chol"], "fasting blood sugar above 120": _h["fbs"],
                     "resting ECG": _h["restecg"], "maximum heart rate": _h["thalach"],
                     "exercise angina": _h["exang"], "ST depression": _h["oldpeak"], "ST slope": _h["slope"],
                     "vessels on fluoroscopy": _h["ca"], "thallium scan": _h["thal"],
                     "disease": _h["disease"]}).reset_index(drop=True)
d = data
used = ["age", "sex", "chest pain type", "resting blood pressure", "cholesterol",
          "maximum heart rate", "exercise angina", "ST depression", "disease"]
missing = d.drop(columns="patient").isna().sum()
print(pd.DataFrame({"missing values": missing, "used by the project": [c in used for c in missing.index]}))
print("patients", len(d), "| with disease", int(d["disease"].sum()), "| share", round(float(d["disease"].mean()), 3))
answer = int(missing[[c for c in missing.index if c not in used]].sum())
teach.check_step("Prompt 1.1", "answer", globals())

**What the result shows.** The 6 missing values all sit in the vessels on fluoroscopy and the thallium scan. Both come from tests that follow the exercise test, so the project leaves them out for a reason of timing, not of missing values: A model with them would use information that the referral does not have yet.

#### Prompt 1.2: How much data the model needs

**Goal.** Compute the three minimum sizes of Riley and colleagues for a model of ten parameters and compare them with the training patients.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The full model of the project has ten parameters: seven variables and three columns for the chest pain type. The cardiologist wants to know whether 227 training patients are enough, and takes the R squared of the reference of age, sex and chest pain type as the anticipated value.  
> Data: The notebook holds a table named "data" with the 227 training patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, and "probability, reference" the probability of coronary artery disease that a reference model of the course, a logistic regression on the age, the sex and the chest pain type, gives the patient after it was fitted on these same patients. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Compute the Cox-Snell R squared of the probabilities: Take the log-likelihood of the probabilities, the sum over the patients of the natural logarithm of the probability given to the outcome that the patient had, and the log-likelihood of a model that gives every patient the overall share of disease; the R squared is one minus the exponential of two times the difference, null minus model, divided by the number of patients. Then compute the three minimum sizes of Riley and colleagues for a model with 10 parameters. The largest R squared that the share of disease allows is one minus the exponential of two times the log-likelihood of the overall share, divided by the number of patients. Size (i), for a shrinkage of 0.9, is 10 divided by (0.9 minus 1) times the natural logarithm of one minus the R squared divided by 0.9. Size (ii) uses the same formula with the shrinkage replaced by the R squared divided by the sum of the R squared and 0.05 times the largest R squared. Size (iii) is (1.96 divided by 0.05) squared, times the share of disease, times one minus the share. Round every size up to a whole patient.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the Cox-Snell R squared, the largest R squared, the three sizes and the number of training patients, and keep size (i) under the name "answer".  
> Test: The R squared must lie between 0 and the largest R squared, and size (iii) must not depend on the probabilities.  

In [ ]:
# Data of Prompt 1.2: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_train"].index, "disease": _c["y_train"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref_train"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.2
teach.check_step("Prompt 1.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.2 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_train"].index, "disease": _c["y_train"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref_train"], 6)})
y = data["disease"].to_numpy(dtype=float)
p = data["probability, reference"].to_numpy(dtype=float)
n, phi, P = len(y), float(y.mean()), 10
ll_null = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
ll_model = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
r2 = 1 - np.exp(2 * (ll_null - ll_model) / n)
r2_max = 1 - np.exp(2 * ll_null / n)
n1 = math.ceil(P / ((0.9 - 1) * np.log(1 - r2 / 0.9)))
s2 = r2 / (r2 + 0.05 * r2_max)
n2 = math.ceil(P / ((s2 - 1) * np.log(1 - r2 / s2)))
n3 = math.ceil((1.96 / 0.05) ** 2 * phi * (1 - phi))
print("Cox-Snell R squared", round(float(r2), 3), "| largest R squared", round(float(r2_max), 3))
print("sizes (i), (ii), (iii):", n1, n2, n3, "| training patients", n)
answer = int(n1)
teach.check_step("Prompt 1.2", "answer", globals())

**What the result shows.** Criterion (i) asks for 241 patients and criterion (iii) for even more, against 227. The project goes on, but its report says that the data are smaller than the criteria ask for, and the calibration slope of Stop 2 will show the mild overfitting that follows.

#### Prompt 1.3: The reference on the test patients

**Goal.** Fit the reference model of age, sex and chest pain type on the training patients and test it once on the test patients.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before the cardiologist asks whether the results of the exercise test help, the project needs the bar to clear: what a cardiologist knows before the test, the age, the sex and the type of chest pain.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina) and "ST depression" (the depression of the ST segment during exercise compared with rest, mm). "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Make the inputs of the reference: the age, a column that is 1 for men and 0 for women, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, so that typical angina is the reference category. Turn each input into a z-score with the mean and the standard deviation of the training patients, and fit the usual logistic regression for disease, with its standard settings and at most 1,000 iterations, on the training patients only. Compute the probability of disease for every test patient and the area under the ROC curve (AUC) on the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of training and test patients and the test AUC of the reference, and keep the AUC, with three decimals, under the name "answer".  
> Test: The model must be fitted on 227 training patients only, and the AUC must lie between 0.5 and 1.  

In [ ]:
# Data of Prompt 1.3: the table "data" that the prompt describes
data = _patients("both")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.3
teach.check_step("Prompt 1.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.3 (run it only if the code of the assistant does not pass the check)
data = _patients("both")
d = data
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int)})
for c in ["atypical angina", "non-anginal pain", "no pain"]:
    X[c] = (d["chest pain type"] == c).astype(int)
tr_ = (d["set"] == "training").to_numpy()
y = d["disease"].to_numpy()
reference = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])
p = reference.predict_proba(X[~tr_])[:, 1]
print("training patients", int(tr_.sum()), "| test patients", int((~tr_).sum()))
print("test AUC of the reference", round(float(roc_auc_score(y[~tr_], p)), 3))
answer = round(float(roc_auc_score(y[~tr_], p)), 3)
teach.check_step("Prompt 1.3", "answer", globals())

**What the result shows.** The reference reaches a test AUC of 0.900 with three variables that every cardiologist knows before the test. Any gain of the full model is measured against this number, in Stop 2.

**«Stop 1 · Prompt and application» is complete.** Next comes «Stop 1 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#model-training-optimisation-and-xai-integration) and continue reading at «Model training, optimisation and XAI integration».

### Stop 1 · Ready application

This part takes the worked project from its question to its model: It audits the Cleveland data, splits them by patient, checks their size with the criteria of Riley and colleagues, chooses the family of the model on the training patients and compares the model with a simple reference.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 1 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week11 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import math
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

#### The question and the audit of the data

*Lecture page: [The data: Source, permission and audit](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#the-data-source-permission-and-audit)*

The worked final project asks whether routine variables known after an exercise test can rule out coronary artery disease and spare a referral for angiography. Its data are the 303 Cleveland patients of Week 1. The audit counts the missing values of every column and marks the columns that the project uses: the eight variables known at the moment of the decision and the outcome.

**From the lecture to the code.** The eight variables of the project: `FEATURES`. The missing values of each column: `missing`. The table of the audit: `audit`.

In [ ]:
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
missing = heart.isna().sum()
audit = pd.DataFrame({"missing values": missing, "used by the project": [c in FEATURES for c in missing.index]})

print(audit)
print(len(heart), "patients,", int(heart["disease"].sum()), "with disease")

**Python note.** `heart.isna()` gives a table of the same shape with `True` where a value is missing, and `.sum()` counts the `True` values of every column. The *list comprehension* `[c in FEATURES for c in missing.index]` asks for every column whether it belongs to the project; `pd.DataFrame({...})` puts both columns side by side, with the names of the columns of the data as its index.

In [ ]:
share = heart.groupby("cp")["disease"].agg(["count", "mean"])
share.index = ["typical angina", "atypical angina", "non-anginal pain", "no pain"]
print(share.round(3))
fig, ax = plt.subplots(figsize=(7.2, 2.8))
ax.barh(share.index, share["mean"], color="#1F5F8B")
ax.set_xlim(0, 1)
ax.set_xlabel("share with disease on the angiogram")
ax.set_title("Disease by chest pain type, 303 Cleveland patients (UCI Heart Disease, CC BY 4.0)", fontsize=9)
plt.tight_layout()
plt.show()

**Observe and reflect.** In these data, the patients without chest pain have the highest share of disease and the patients with typical angina one of the lowest, the opposite of what the pretest probability of a chest pain clinic expects. What does this say about how these patients came to angiography, and what would a model that learned it do in your clinic?

**Exercise A.** What share of the 303 patients has disease? Keep it with three decimals.

In [ ]:
data = exercise_table("A", heart=heart)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(heart["disease"].mean()), 3)
teach.check("Exercise A", answer)

#### The split by patient

*Lecture page: [The data: Source, permission and audit](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#the-data-source-permission-and-audit)*

The project keeps the split of Week 2: a quarter of the patients for the test, drawn with the same share of disease in both sets. Every row of the Cleveland data is one patient, so a split of the rows is a split by patient; with several rows per patient, all rows of a patient would have to stay in one set. The chest pain type becomes three columns of 0 and 1, with typical angina as the reference category.

**From the lecture to the code.** The inputs and the outcome: `X`, `y`. The training and the test patients: `X_train`, `X_test`, `y_train`, `y_test`.

In [ ]:
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print(pd.DataFrame({"patients": [len(y_train), len(y_test)], "with disease": [int(y_train.sum()), int(y_test.sum())]},
                   index=["training", "test"]))
print(list(X.columns))

**Python note.** `pd.get_dummies(..., columns=["cp"], drop_first=True, dtype=int)` turns the chest pain type, coded 1 to 4, into the columns `cp_2.0`, `cp_3.0` and `cp_4.0`; `drop_first=True` leaves out the first category, typical angina, which becomes the reference. `train_test_split(..., stratify=y, random_state=RANDOM_STATE)` draws the test quarter with the share of disease of the whole data in both sets and with the seed 2026 of the course, so that the split is the same in every week and on every computer.

**Observe and reflect.** The test set holds 76 patients, 35 of them with disease. The section on the bootstrap of the lecture page shows that a sensitivity can only have a lower bound of 0.95 when at least 73 patients with disease are all found. What does this mean for what the test set of this project can show?

**Exercise B.** How many of the test patients have disease?

In [ ]:
data = exercise_table("B", X_test=X_test, X_train=X_train, heart=heart)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(y_test.sum())
teach.check("Exercise B", answer)

#### How much data

*Lecture page: [How much data: The criteria of Riley and colleagues](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#how-much-data-the-criteria-of-riley-and-colleagues)*

The three criteria of Riley and colleagues give the smallest number of patients for a model of a binary outcome. They need the number of parameters, the share of the outcome and an anticipated Cox-Snell R squared, which the project takes from the reference of age, sex and chest pain type on the training patients.

**From the formulas to the code.** $R^2_{CS}$: `cox_snell_r2`. $n_{(i)}$, $n_{(ii)}$, $n_{(iii)}$ and $R^2_{\max}$: `riley_sizes`. $P$: `P`. The anticipated $R^2_{CS}$: `r2_cs`.

**Start cell.** For the section «How much data», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «How much data»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

**On the lecture page:** [Worked example: The three sizes for the worked project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#worked-example-the-three-sizes-for-the-worked-project). The cell below computes the same three sizes from the training patients.

In [ ]:
def cox_snell_r2(y, p):
    """The Cox-Snell R squared of predicted probabilities p for the outcomes y."""
    y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
    phi = y.mean()
    ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
    ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
    """The three sizes of Riley and colleagues for a model of a binary outcome with P parameters: (i) a shrinkage of at least 0.9, (ii) apparent and adjusted Nagelkerke R squared within 0.05, (iii) the overall risk within 0.05."""
    r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
    n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
    s2 = r2_cs / (r2_cs + delta * r2_max)
    n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
    n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
    return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
P = X_train.shape[1]
sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

print("P =", P, "| prevalence", round(float(y_train.mean()), 3), "| R² (Cox-Snell)", round(float(r2_cs), 3))
print(sizes, "| training patients", len(y_train))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#nbref-c128)*

**Python note.** `cox_snell_r2` compares two log-likelihoods: that of the probabilities `p` and that of a model that gives every patient the overall share `phi`; `np.clip` keeps every probability away from exactly 0 and 1, where the logarithm would be infinite. `riley_sizes` writes the three formulas of the lecture line by line, and `math.ceil` rounds each size up to a whole patient. Its arguments `shrinkage`, `delta` and `margin` have *default values*, so `riley_sizes(P, prevalence, r2_cs)` uses 0.9, 0.05 and 0.05 unless other values are given.

In [ ]:
grid = np.linspace(0.05, 0.6, 56)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for key, colour in (("(i)", "#9B1B1B"), ("(ii)", "#B97813"), ("(iii)", "#0E7A78")):
    ax.plot(grid, [riley_sizes(P, float(y_train.mean()), r)[key] for r in grid], color=colour, label=key)
ax.axhline(len(y_train), color="#231A1A", linestyle="--", linewidth=1, label="training patients")
ax.axvline(r2_cs, color="#8A8A8A", linestyle=":", linewidth=1)
ax.set_ylim(0, 800)
ax.set_xlabel("anticipated Cox-Snell R squared")
ax.set_ylabel("patients needed")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Exercise C.** How many patients would criterion (i) ask for if the model had only the five parameters of the reference, with the same R squared and share of disease?

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = riley_sizes(5, float(y_train.mean()), r2_cs)["(i)"]
teach.check("Exercise C", answer)

#### A reference and the model

*Lecture page: [A reference first](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#a-reference-first)*

The reference is a logistic regression on what the cardiologist knows before the exercise test: the age, the sex and the chest pain type. The full model adds the resting blood pressure, the cholesterol and the three results of the test. Both are fitted on the training patients with the same pipeline, and both are tested once on the same test patients.

**From the lecture to the code.** The columns of the reference and of the model: `REFERENCE`, `ALL`. The two fitted models: `reference`, `model`. Their probabilities for the test patients: `p_ref`, `p_model`. The outcomes of the test patients as an array: `yt`.

**Start cell.** For the section «A reference and the model», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «A reference and the model»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week11 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]

**First, the choice of the model, on the training patients only.** Two families of models compete: the logistic regression with five strengths of its regularisation, and a random forest with eight settings of its leaves and of the variables tried at each split. Five-fold cross-validation scores every candidate on the training patients; the test patients stay closed. The cell takes about a minute.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "logistic regression": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                                                {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}),
    "random forest": (RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
                                       {"min_samples_leaf": [1, 5, 10, 20], "max_features": ["sqrt", None]}),
}
rows = []
for name, (estimator, grid) in candidates.items():
    search = GridSearchCV(estimator, grid, scoring="roc_auc", cv=folds).fit(X_train, y_train)
    for settings, mean, sd in zip(search.cv_results_["params"], search.cv_results_["mean_test_score"], search.cv_results_["std_test_score"]):
        rows.append({"model": name, "settings": ", ".join(f"{k.split('__')[-1]} = {v}" for k, v in settings.items()),
                     "AUC, 5-fold": round(float(mean), 3), "standard error": round(float(sd) / np.sqrt(5), 3)})
selection = pd.DataFrame(rows).sort_values("AUC, 5-fold", ascending=False)

print(selection.head(8).to_string(index=False))
best = selection.groupby("model")["AUC, 5-fold"].max()
print(best.round(3).to_dict())

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#nbref-c138)*

**Python note.** `GridSearchCV(estimator, grid, scoring="roc_auc", cv=folds)` fits every combination of the settings in `grid` on four fifths of the training patients and scores it on the fifth, five times, and `cv_results_` keeps the mean and the standard deviation of the five AUCs. The standard error of the mean is the standard deviation divided by the square root of 5. `C` is the inverse strength of the regularisation of the logistic regression: a small `C` pulls the coefficients towards zero. `min_samples_leaf` is the smallest number of patients in a leaf of a tree, and `max_features` the number of variables tried at each split, the square root of their number or all of them.

In [ ]:
ALL = list(X.columns)


def fit(columns):
    """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


reference, model = fit(REFERENCE), fit(ALL)
p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
p_model = model.predict_proba(X_test[ALL])[:, 1]
yt = y_test.to_numpy()

print("test AUC, reference", round(roc_auc_score(yt, p_ref), 3), "| test AUC, model", round(roc_auc_score(yt, p_model), 3))
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for name, p, colour in (("reference", p_ref, "#8A8A8A"), ("model", p_model, "#9B1B1B")):
    fpr, tpr, _ = roc_curve(yt, p)
    ax.plot(fpr, tpr, color=colour, label=f"{name}, AUC {roc_auc_score(yt, p):.3f}")
ax.plot([0, 1], [0, 1], color="#C8C0B8", linewidth=1)
ax.set_xlabel("1 − specificity")
ax.set_ylabel("sensitivity")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** `fit(columns)` is a small function that returns a fitted pipeline, so the reference and the model are built by the same line with two lists of columns. `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` learns the means and the standard deviations of the columns from the training patients when it is fitted, and applies them to the test patients when it predicts. `predict_proba(...)[:, 1]` takes the second column of the probabilities, that of disease.

**Observe and reflect.** The two ROC curves cross and touch in several places. Could you tell from the curves alone whether the full model is better than the reference on new patients?

**Exercise D.** What test AUC does a model of the reference columns and the three results of the exercise test reach, without the blood pressure and the cholesterol? Keep it with three decimals.

In [ ]:
data = exercise_table("D", X_test=X_test, X_train=X_train, heart=heart)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
exercise_only = fit(REFERENCE + ["thalach", "exang", "oldpeak"])
answer = round(float(roc_auc_score(yt, exercise_only.predict_proba(X_test[REFERENCE + ["thalach", "exang", "oldpeak"]])[:, 1])), 3)
teach.check("Exercise D", answer)

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#model-training-optimisation-and-xai-integration) and continue reading at «Model training, optimisation and XAI integration». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: Uncertainty and calibration

*Lecture page: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#stop-2)*

This stop asks how sure the results are. The bootstrap gives the intervals of the AUCs of the reference and of the full model, and of their difference in the same resamples; the calibration slope asks whether the probabilities of the full model hold on the test patients.

**At the end of this stop:** The lower bound of the test AUC of the full model (0.872), the lower bound of its difference from the reference (−0.020) and the calibration slope on the test patients (1.241).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 2 · Python basics

*Lecture page: [Uncertainty: The bootstrap](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#uncertainty-the-bootstrap)*

The ready application of this stop gives every result of the final project an interval: It resamples the test patients, compares the AUCs of the reference and of the model in the same resamples and checks the calibration of the probabilities. The examples below introduce these constructs on ten patients: a percentile interval, one resample for two models, many resamples and the calibration slope.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 2 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
# Python practice: a small example for learning the language
aucs = np.array([0.70, 0.74, 0.68, 0.81, 0.77, 0.73, 0.79, 0.71, 0.76, 0.75])   # the AUCs of ten resamples
print(np.percentile(aucs, [2.5, 97.5]).round(3))

**Python note.** `np.percentile(aucs, [2.5, 97.5])` gives the values below which 2.5 and 97.5 percent of the AUCs lie, interpolated between neighbouring values. Between them lie 95 percent of the AUCs: the percentile interval of the bootstrap. The ready application computes it from 2,000 resamples, not ten.

In [ ]:
# Python practice: a small example for learning the language
y = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])                              # the outcomes of ten test patients
p_a = np.array([0.2, 0.6, 0.5, 0.7, 0.3, 0.4, 0.45, 0.8, 0.35, 0.25])   # the reference
p_b = np.array([0.1, 0.3, 0.7, 0.9, 0.2, 0.6, 0.5, 0.85, 0.4, 0.45])    # the model
print("AUC", roc_auc_score(y, p_a), roc_auc_score(y, p_b))
rng = np.random.default_rng(2026)
i = rng.integers(0, len(y), len(y))
print(i, y[i], 0 < y[i].sum() < len(i))

**Python note.** `rng.integers(0, len(y), len(y))` draws as many positions as there are patients, with replacement, so a patient may come twice or not at all. The same positions `i` select the outcomes and the probabilities of both models, `y[i]`, `p_a[i]` and `p_b[i]`: The two models are compared on the same resampled patients, a *paired* comparison. An AUC needs patients with and without disease, so `0 < y[i].sum() < len(i)` checks that the resample holds both.

In [ ]:
# Python practice: a small example for learning the language
differences = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    if 0 < y[i].sum() < len(i):
        differences.append(roc_auc_score(y[i], p_b[i]) - roc_auc_score(y[i], p_a[i]))
differences = np.array(differences)
print(len(differences), np.percentile(differences, [2.5, 97.5]).round(3), round(float((differences <= 0).mean()), 3))

**Python note.** The loop repeats the resample 1,000 times and keeps the difference of the two AUCs whenever the resample holds both kinds of patients. The model is better by 0.12 on the ten patients, but the interval of the difference runs from below zero to far above it, and in about three resamples of ten the model is not better at all: Ten patients cannot decide between the two models. The function `bootstrap_auc` of the ready application does the same with the 76 test patients of the project.

In [ ]:
# Python practice: a small example for learning the language
p = np.array([0.1, 0.3, 0.6, 0.8, 0.2, 0.7, 0.4, 0.9, 0.5, 0.35])
outcome = np.array([0, 1, 1, 1, 0, 0, 0, 1, 1, 0])
z = np.log(p / (1 - p)).reshape(-1, 1)   # the logits as one column
slope = LogisticRegression(C=1e6).fit(z, outcome).coef_[0, 0]
print(z.shape, round(float(slope), 3))

**Python note.** The calibration slope is the coefficient of a logistic regression of the outcomes on the logits of the model. `reshape(-1, 1)` turns the ten logits into a table of ten rows and one column, the form that scikit-learn expects for its inputs, and `C=1e6` makes the regularisation so weak that it does not matter. A slope of 1 means that the probabilities spread as much as the outcomes ask; below 1 they are too extreme, above 1, as here, too modest. The function `calibration` of the ready application finds the same slope by Newton steps.

**Exercise 2.1.** Ten bootstrap differences of the AUC are −0.02, 0.05, 0.01, −0.01, 0.03, 0.04, 0.00, 0.06, 0.02 and 0.03. In what share of them is the model not better than the reference, a difference of 0 or less?

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d = np.array([-0.02, 0.05, 0.01, -0.01, 0.03, 0.04, 0.00, 0.06, 0.02, 0.03])
answer = float((d <= 0).mean())
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** The AUCs of five resamples are 0.80, 0.84, 0.78, 0.86 and 0.82. What is their 50th percentile, `np.percentile(..., 50)`?

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = float(np.percentile([0.80, 0.84, 0.78, 0.86, 0.82], 50))
teach.check("Exercise 2.2", answer)

**«Stop 2 · Python basics» is complete.** Next comes «Stop 2 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity».

### Stop 2 · Prompt and application

This part has three prompts: «The intervals of the two AUCs», «The interval of the gain» and «Calibration on the test patients». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-11/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 2 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _course():
    """The worked project of the week: the Cleveland data, the split of Week 2, the reference, the full model and the cross-validated probabilities of the training patients (computed once, a few seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland(source=DATA_SOURCE)
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"missing values": missing, "used by the project": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """The Cox-Snell R squared of predicted probabilities p for the outcomes y."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """The three sizes of Riley and colleagues for a model of a binary outcome with P parameters: (i) a shrinkage of at least 0.9, (ii) apparent and adjusted Nagelkerke R squared within 0.05, (iii) the overall risk within 0.05."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """The training or the test patients, or both in the order of the split, with their set, the eight clinical variables and the disease."""
    c = _course()
    parts = {"training": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"patient": h.index, "set": [s for s, _ in rows],
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "chest pain type": h["cp"].map(_CP),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "disease": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "patients of the Cleveland data:", len(_c["X_train"]), "for training and", len(_c["X_test"]), "for the test")

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.metrics import roc_auc_score, roc_curve

#### Prompt 2.1: The intervals of the two AUCs

**Goal.** Give the test AUCs of the reference and of the full model with their bootstrap intervals.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The report of the final project gives every result with its uncertainty. With 76 test patients, the cardiologist wants to know how far each AUC could move with other patients of the same kind.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, "probability, reference" the probability of coronary artery disease from a reference model of the course (a logistic regression on the age, the sex and the chest pain type), and "probability, model" the probability from the full model of the course (the same with the resting blood pressure, the cholesterol and three results of the exercise test added). Both models were fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Draw 2,000 resamples of the test patients: Each resample draws as many patients as the table has, with replacement, with a random number generator started with the seed 2026. Skip a resample that holds patients of one outcome only. In each resample, compute the area under the ROC curve (AUC) of the reference and of the model on the same patients, and the difference, model minus reference. The 95 percent interval of a result runs from the 2.5th to the 97.5th percentile of its values over the resamples.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of resamples used and a table with the AUC of each model and of the difference on all test patients, with their 2.5th and 97.5th percentiles, and keep the 2.5th percentile of the AUC of the model, with three decimals, under the name "answer".  
> Test: Each interval must contain the AUC of all test patients, and the same resamples must serve both models.  

In [ ]:
# Data of Prompt 2.1: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref"], 6), "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.1
teach.check_step("Prompt 2.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.1 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref"], 6), "probability, model": np.round(_c["p_model"], 6)})
y = data["disease"].to_numpy()
p_ref = data["probability, reference"].to_numpy()
p_model = data["probability, model"].to_numpy()
rng = np.random.default_rng(2026)
rows = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if y[i].min() == y[i].max():
        continue
    a, b = roc_auc_score(y[i], p_ref[i]), roc_auc_score(y[i], p_model[i])
    rows.append({"reference": a, "model": b, "difference": b - a})
boot = pd.DataFrame(rows)
table = pd.DataFrame({"AUC": [roc_auc_score(y, p_ref), roc_auc_score(y, p_model), roc_auc_score(y, p_model) - roc_auc_score(y, p_ref)],
                      "2.5 %": boot.quantile(0.025).to_numpy(), "97.5 %": boot.quantile(0.975).to_numpy()}, index=boot.columns).round(3)
print(len(boot), "resamples")
print(table)
answer = round(float(np.percentile(boot["model"], 2.5)), 3)
teach.check_step("Prompt 2.1", "answer", globals())

**What the result shows.** The AUC of the full model has the lower bound 0.872: With 76 patients, the interval of each model spans about a tenth of the scale of the AUC.

#### Prompt 2.2: The interval of the gain

**Goal.** Give the difference between the full model and the reference with its bootstrap interval, computed on the same resamples.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The question of the project is whether the results of the exercise test add to what the cardiologist knew before the test. The answer is the interval of the difference, not the two AUCs side by side.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, "probability, reference" the probability of coronary artery disease from a reference model of the course (a logistic regression on the age, the sex and the chest pain type), and "probability, model" the probability from the full model of the course (the same with the resting blood pressure, the cholesterol and three results of the exercise test added). Both models were fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Draw 2,000 resamples of the test patients: Each resample draws as many patients as the table has, with replacement, with a random number generator started with the seed 2026. Skip a resample that holds patients of one outcome only. In each resample, compute the area under the ROC curve (AUC) of the reference and of the model on the same patients, and the difference, model minus reference. The 95 percent interval of a result runs from the 2.5th to the 97.5th percentile of its values over the resamples. Also give the share of the resamples in which the difference is 0 or less.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the table of the three results with their intervals and the share of resamples in which the model is not better, and keep the 2.5th percentile of the difference, with three decimals, under the name "answer".  
> Test: The difference must be computed in each resample from the two AUCs of that resample, not from two separate sets of resamples.  

In [ ]:
# Data of Prompt 2.2: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref"], 6), "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.2
teach.check_step("Prompt 2.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.2 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, reference": np.round(_c["p_ref"], 6), "probability, model": np.round(_c["p_model"], 6)})
y = data["disease"].to_numpy()
p_ref = data["probability, reference"].to_numpy()
p_model = data["probability, model"].to_numpy()
rng = np.random.default_rng(2026)
rows = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if y[i].min() == y[i].max():
        continue
    a, b = roc_auc_score(y[i], p_ref[i]), roc_auc_score(y[i], p_model[i])
    rows.append({"reference": a, "model": b, "difference": b - a})
boot = pd.DataFrame(rows)
table = pd.DataFrame({"AUC": [roc_auc_score(y, p_ref), roc_auc_score(y, p_model), roc_auc_score(y, p_model) - roc_auc_score(y, p_ref)],
                      "2.5 %": boot.quantile(0.025).to_numpy(), "97.5 %": boot.quantile(0.975).to_numpy()}, index=boot.columns).round(3)
print(len(boot), "resamples")
print(table)
print("share of resamples in which the model is not better:", round(float((boot["difference"] <= 0).mean()), 3))
answer = round(float(np.percentile(boot["difference"], 2.5)), 3)
teach.check_step("Prompt 2.2", "answer", globals())

**What the result shows.** The interval of the difference starts at −0.020, below zero: On these data, the full model is not clearly better than the reference. The report says so, with the interval.

#### Prompt 2.3: Calibration on the test patients

**Goal.** Compute the calibration intercept and slope of the full model on the test patients.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A rule-out uses the probabilities themselves. The cardiologist wants to know whether they are right on average and whether they are too extreme or too modest.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, and "probability, model" the probability of coronary artery disease from a logistic regression of the course on eight clinical variables, fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Turn every probability into its log-odds, the natural logarithm of the probability divided by one minus the probability. The calibration intercept comes from a logistic regression for disease with only an intercept and the log-odds as a fixed offset; the calibration slope is the coefficient of the log-odds in a logistic regression for disease with an intercept and the log-odds as its only input. Fit both without any penalty, for example by Newton steps.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean probability, the observed share of disease, the calibration intercept and the calibration slope, and keep the slope, with three decimals, under the name "answer".  
> Test: A model that is perfectly calibrated would give an intercept of 0 and a slope of 1; the intercept must have the sign of the observed share minus the mean probability.  

In [ ]:
# Data of Prompt 2.3: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.3
teach.check_step("Prompt 2.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.3 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, model": np.round(_c["p_model"], 6)})
y = data["disease"].to_numpy(dtype=float)
p = data["probability, model"].to_numpy(dtype=float)
lp = np.log(p / (1 - p))
a = 0.0
for _ in range(50):
    q = 1 / (1 + np.exp(-(a + lp)))
    a += (y - q).sum() / (q * (1 - q)).sum()
Z = np.c_[np.ones_like(lp), lp]
b = np.zeros(2)
for _ in range(50):
    q = 1 / (1 + np.exp(-Z @ b))
    b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
print("mean probability", round(float(p.mean()), 3), "| observed share", round(float(y.mean()), 3))
print("calibration intercept", round(float(a), 3), "| calibration slope", round(float(b[1]), 3))
answer = round(float(b[1]), 3)
teach.check_step("Prompt 2.3", "answer", globals())

**What the result shows.** The slope is 1.241 on 76 patients, with an interval that includes 1. The cross-validated probabilities of the training patients give a slope a little below 1, the mild overfitting that criterion (i) expected.

**«Stop 2 · Prompt and application» is complete.** Next comes «Stop 2 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity».

### Stop 2 · Ready application

This part gives the AUCs of the reference and of the model and their difference with intervals from the bootstrap of the test patients, and checks the calibration of both.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 2 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week11 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
ALL = list(X.columns)
def fit(columns):
    """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)
reference, model = fit(REFERENCE), fit(ALL)
p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
p_model = model.predict_proba(X_test[ALL])[:, 1]
yt = y_test.to_numpy()

#### Uncertainty: The bootstrap

*Lecture page: [Uncertainty: The bootstrap](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#uncertainty-the-bootstrap)*

The bootstrap draws the test patients again with replacement, computes the AUCs of both models on the same resample, and repeats this 2,000 times. The 2.5th and the 97.5th percentiles of the results give the 95 percent intervals; the difference of the two AUCs is computed in each resample, so that its interval answers the question of the project. The Wilson interval gives the uncertainty of a proportion.

**From the formulas to the code.** One resample: `i`. The AUCs and their difference in every resample: `boot`. The percentile interval $[q_{0.025}, q_{0.975}]$: `interval`. The Wilson interval: `wilson`.

**On the lecture page:** [Worked example: The bootstrap of ten patients](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#worked-example-the-bootstrap-of-ten-patients). The cell below repeats the bootstrap with the 76 test patients.

In [ ]:
def bootstrap_auc(y, p_a, p_b, n_boot=2000, seed=RANDOM_STATE):
    """AUCs of two models and their difference in n_boot resamples of the test patients, drawn with replacement."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    rows = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        a, b = roc_auc_score(y[i], p_a[i]), roc_auc_score(y[i], p_b[i])
        rows.append({"a": a, "b": b, "difference": b - a})
    return pd.DataFrame(rows)


def interval(values, level=0.95):
    """The percentile interval of the bootstrap values."""
    return tuple(np.percentile(values, [50 * (1 - level), 100 - 50 * (1 - level)]))


def wilson(k, n, z=1.96):
    """The Wilson interval of a proportion k/n."""
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half

boot = bootstrap_auc(yt, p_ref, p_model)
print(len(boot), "resamples")
for column in ["a", "b", "difference"]:
    low_, high_ = interval(boot[column])
    print(f"{column:>10}: {boot[column].mean():.3f} [{low_:.3f}, {high_:.3f}]")
print("share in which the model is not better", round(float((boot["difference"] <= 0).mean()), 3))
print("sensitivity 35 of 35, Wilson", np.round(wilson(35, 35), 3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#nbref-c150)*

**Python note.** Inside `bootstrap_auc`, `rng.integers(0, len(y), len(y))` draws as many positions as there are patients, with replacement, and `y[i]`, `p_a[i]` and `p_b[i]` pick the same patients from all three arrays: The two models are always judged on the same resample. A resample in which all patients have the same outcome has no AUC, so `if y[i].min() == y[i].max(): continue` skips it. `rows.append({...})` collects one dictionary per resample, and `pd.DataFrame(rows)` makes the table `boot` with the columns `a`, `b` and `difference`. `np.percentile(values, [2.5, 97.5])` gives the two ends of the interval.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2))
axes[0].hist(boot["a"], bins=40, color="#8A8A8A", alpha=0.7, label="reference")
axes[0].hist(boot["b"], bins=40, color="#9B1B1B", alpha=0.6, label="model")
axes[0].set_xlabel("AUC")
axes[0].legend(fontsize=8)
axes[1].hist(boot["difference"], bins=40, color="#1F5F8B")
for v in interval(boot["difference"]):
    axes[1].axvline(v, color="#231A1A", linestyle="--", linewidth=1)
axes[1].axvline(0, color="#231A1A", linewidth=1.2)
axes[1].set_xlabel("AUC of the model minus AUC of the reference")
plt.tight_layout()
plt.show()

**Observe and reflect.** The histogram of the difference has a part to the left of zero. How would you say in one sentence of the report what the final project shows about the results of the exercise test, and what it does not show?

**Advanced (optional).** Run `bootstrap_auc(yt, p_ref, p_model, seed=1)` and compare its intervals with those of the seed 2026. The intervals move in the third decimal: The seed decides which resamples are drawn, but the width of the interval comes from the 76 patients.

**Exercise E.** In what share of the resamples is the full model not better than the reference? Keep it with three decimals.

In [ ]:
data = exercise_table("E", X_test=X_test, p_model=p_model, p_ref=p_ref, yt=yt)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float((boot["difference"] <= 0).mean()), 3)
teach.check("Exercise E", answer)

#### Calibration

*Lecture page: [Calibration: Do the probabilities hold?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#calibration-do-the-probabilities-hold)*

Two logistic regressions of the outcomes on the log-odds of the model give the calibration intercept, with the log-odds as a fixed offset, and the calibration slope. A few Newton steps fit them without any penalty. The bootstrap gives the interval of the slope on the 76 test patients.

**From the formulas to the code.** $\mathrm{LP}$: `lp`. The intercept $a$ and the slope $b$: the two values that `calibration` returns. The interval of the slope: `slope_interval`.

In [ ]:
def calibration(y, p, steps=50):
    """Calibration intercept (with the linear predictor as offset) and slope of predicted probabilities p, by Newton steps."""
    y = np.asarray(y, dtype=float)
    lp = np.log(p / (1 - p))
    a = 0.0
    for _ in range(steps):
        q = 1 / (1 + np.exp(-(a + lp)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    Z = np.c_[np.ones_like(lp), lp]
    b = np.zeros(2)
    for _ in range(steps):
        q = 1 / (1 + np.exp(-Z @ b))
        b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
    return float(a), float(b[1])


def slope_interval(y, p, n_boot=1000, seed=RANDOM_STATE):
    """The bootstrap percentile interval of the calibration slope."""
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    slopes = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() < y[i].max():
            slopes.append(calibration(y[i], p[i])[1])
    return interval(slopes)

a_test, b_test = calibration(yt, p_model)
print("test: intercept", round(a_test, 3), "| slope", round(b_test, 3), "| interval of the slope", np.round(slope_interval(yt, p_model), 3))
print("mean probability", round(float(p_model.mean()), 3), "| observed share", round(float(yt.mean()), 3))
groups = pd.qcut(p_model, 5, labels=False)
fifths = pd.DataFrame({"predicted": pd.Series(p_model).groupby(groups).mean(), "observed": pd.Series(yt).groupby(groups).mean(),
                       "patients": pd.Series(yt).groupby(groups).size()}).round(3)
print(fifths)
fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.plot([0, 1], [0, 1], color="#C8C0B8", linewidth=1)
ax.plot(fifths.iloc[:, 0], fifths.iloc[:, 1], "o-", color="#9B1B1B")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed share with disease")
plt.tight_layout()
plt.show()

**Python note.** The intercept is fitted first: `q` are the probabilities with the shift `a`, and each step adds the sum of the residuals `y - q` divided by the sum of the weights `q * (1 - q)`. For the slope, `Z` holds a column of ones and the log-odds, and each Newton step solves a small system of two equations with `np.linalg.solve`. Fifty steps are far more than needed; the steps stop moving after a handful. `pd.qcut(p_model, 5, labels=False)` cuts the patients into five groups of the same size by their probability.

**Observe and reflect.** The interval of the slope on the test patients runs from below 1 to above 2. Which number would you put in the report as the calibration of the model, and with which words?

**Exercise F.** What is the calibration slope of the reference on the test patients? Keep it with three decimals.

In [ ]:
data = exercise_table("F", X_test=X_test, p_model=p_model, p_ref=p_ref, yt=yt)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(calibration(yt, p_ref)[1], 3)
teach.check("Exercise F", answer)

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: The rule-out and its limits

*Lecture page: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#stop-3)*

This stop turns the probabilities into a decision. The rule-out threshold is chosen on the cross-validated probabilities of the training patients, the test patients show how many referrals it would spare, and the women and men of the test set show whether the rule-out can be called safe in both groups.

**At the end of this stop:** The rule-out threshold (0.18), the test patients ruled out (27) and the lower bound of the sensitivity in women (0.646).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 3 · Python basics

*Lecture page: [A decision threshold: Ruling out](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#a-decision-threshold-ruling-out)*

The ready application of this stop chooses a rule-out threshold on probabilities that the training patients get from cross-validation, gives its sensitivity and negative predictive value with Wilson intervals, and reads the coefficients of the model from its pipeline. The examples below introduce these constructs on a few patients: predictions from cross-validation, the largest threshold that keeps a sensitivity, a function for the Wilson interval and the steps of a pipeline.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 3 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Python practice: a small example for learning the language
Xs = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0], [7.0], [8.0]])
ys = np.array([0, 0, 1, 0, 1, 1, 0, 1])
folds = StratifiedKFold(4, shuffle=True, random_state=2026)
p_cv = cross_val_predict(LogisticRegression(), Xs, ys, cv=folds, method="predict_proba")[:, 1]
print(p_cv.round(2))

**Python note.** `cross_val_predict` fits the model four times, each time on three quarters of the patients, and gives every patient the probability of the model that did not see it. These probabilities behave like those of new patients, so a threshold can be chosen on them without touching the test patients. `[:, 1]` keeps the probability of the outcome 1. The ready application chooses its rule-out threshold on the probabilities that five-fold cross-validation gives the training patients.

In [ ]:
# Python practice: a small example for learning the language
p_disease = np.array([0.62, 0.35, 0.81, 0.28, 0.55, 0.91, 0.47, 0.73, 0.66, 0.39])   # ten patients with disease
grid = np.round(np.arange(0.05, 0.51, 0.05), 2)
threshold = max(t for t in grid if (p_disease >= t).mean() >= 0.9)
print(threshold, (p_disease >= threshold).mean())

**Python note.** `(p_disease >= t).mean()` is the share of the patients with disease whose probability reaches the threshold `t`, the sensitivity at `t`. The generator inside `max(...)` keeps the thresholds of the grid at which the sensitivity is at least 0.90, and `max` takes the largest of them, here 0.35. A larger threshold rules out more patients, and the target of the sensitivity stops it before too many patients with disease are missed. The ready application chooses its threshold in this way.

In [ ]:
# Python practice: a small example for learning the language
def wilson(k, n, z=1.96):
    """The 95 percent Wilson interval of a proportion k/n."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


print(np.round(wilson(29, 30), 3), np.round(wilson(35, 35), 3))

**Python note.** `wilson` returns the two ends of the interval as a pair, which `np.round` rounds at once. The Wilson interval stays between 0 and 1 even for a proportion of 1: When all 35 patients with disease are found, the sensitivity of 1 has the lower bound 35/(35 + 1.96²) = 0.901, as in the worked example of the lecture page. The ready application gives the sensitivity and the negative predictive value of the threshold with this function.

In [ ]:
# Python practice: a small example for learning the language
pipe = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xs, ys)
print(list(pipe.named_steps))
print(pipe.named_steps["logisticregression"].coef_.round(3), pipe.named_steps["standardscaler"].scale_.round(3))

**Python note.** `make_pipeline` names its steps after their classes in small letters, and `named_steps` reaches each step by that name. The coefficient of the logistic regression belongs to the standardised column, so dividing it by `scale_` of the scaler gives the effect per unit of the original column. The ready application reads the coefficients of the final model in this way and explains the probability of Patient 260 by the contribution of every variable.

**Exercise 3.1.** The cross-validated probabilities of six patients with disease are 0.62, 0.35, 0.81, 0.28, 0.55 and 0.91. Which is the largest threshold of the grid 0.20, 0.25, ..., 0.50 that keeps the sensitivity at 0.80 or more?

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_disease = np.array([0.62, 0.35, 0.81, 0.28, 0.55, 0.91])
grid = np.round(np.arange(0.20, 0.51, 0.05), 2)
answer = float(max(t for t in grid if (p_disease >= t).mean() >= 0.8))
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** A threshold rules out 30 patients, 29 of them without disease. What is the lower bound of the 95 percent Wilson interval of the negative predictive value? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(wilson(29, 30)[0]), 3)
teach.check("Exercise 3.2", answer)

**«Stop 3 · Python basics» is complete.** Next comes «Stop 3 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity».

### Stop 3 · Prompt and application

This part has three prompts: «The rule-out threshold», «The rule-out on the test patients» and «Women and men». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-11/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 3 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _course():
    """The worked project of the week: the Cleveland data, the split of Week 2, the reference, the full model and the cross-validated probabilities of the training patients (computed once, a few seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland(source=DATA_SOURCE)
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"missing values": missing, "used by the project": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """The Cox-Snell R squared of predicted probabilities p for the outcomes y."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """The three sizes of Riley and colleagues for a model of a binary outcome with P parameters: (i) a shrinkage of at least 0.9, (ii) apparent and adjusted Nagelkerke R squared within 0.05, (iii) the overall risk within 0.05."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """The training or the test patients, or both in the order of the split, with their set, the eight clinical variables and the disease."""
    c = _course()
    parts = {"training": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"patient": h.index, "set": [s for s, _ in rows],
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "chest pain type": h["cp"].map(_CP),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "disease": h["disease"]}).reset_index(drop=True)


_c = _course()
print(len(_c["heart"]), "patients of the Cleveland data:", len(_c["X_train"]), "for training and", len(_c["X_test"]), "for the test")

#### Prompt 3.1: The rule-out threshold

**Goal.** Choose the rule-out threshold on the cross-validated probabilities of the training patients.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A patient below the threshold would not be referred for angiography, so the threshold must keep a very high sensitivity: at least 95 percent of the patients with disease must lie at or above it. The test patients must play no part in the choice.  
> Data: The notebook holds a table named "data" with the 227 training patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, and "probability, cross-validated" the probability of coronary artery disease from a logistic regression of the course on eight clinical variables, computed by five-fold cross-validation: Each patient gets the probability of a model that was fitted on the other four fifths of the training patients. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Among the thresholds 0.01, 0.02 and so on up to 0.50, take the highest at which at least 95 percent of the training patients with disease have a probability at or above it. Give the sensitivity at that threshold and the share of the training patients whose probability lies below it.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the threshold, the sensitivity and the share ruled out on the training patients, and keep the threshold, with two decimals, under the name "answer".  
> Test: The sensitivity at the threshold must be at least 0.95, and at the next threshold of the grid it must be below 0.95.  

In [ ]:
# Data of Prompt 3.1: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_train"].index, "disease": _c["y_train"].to_numpy(),
                     "probability, cross-validated": np.round(_c["p_cv"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.1
teach.check_step("Prompt 3.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.1 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_train"].index, "disease": _c["y_train"].to_numpy(),
                     "probability, cross-validated": np.round(_c["p_cv"], 6)})
y = data["disease"].to_numpy()
p = data["probability, cross-validated"].to_numpy()
grid = np.round(np.arange(0.01, 0.51, 0.01), 2)
threshold = max(t for t in grid if (p[y == 1] >= t).mean() >= 0.95)
print("threshold", threshold, "| sensitivity, training", round(float((p[y == 1] >= threshold).mean()), 3),
      "| share ruled out, training", round(float((p < threshold).mean()), 3))
answer = float(threshold)
teach.check_step("Prompt 3.1", "answer", globals())

**What the result shows.** The threshold is 0.18. It was chosen without a single test patient, so the test result of the next step is an honest estimate for new patients.

#### Prompt 3.2: The rule-out on the test patients

**Goal.** Apply the threshold to the test patients once and give the sensitivity, the specificity and the negative predictive value with their Wilson intervals.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how many referrals the rule-out would spare, whether it misses any patient with disease, and how sure these numbers are with 76 patients.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, and "probability, model" the probability of coronary artery disease from a logistic regression of the course on eight clinical variables, fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise. The rule-out threshold of the project, chosen on the training patients, is 0.18: A patient whose probability lies below it would not be referred for angiography.  
> Task: A patient is ruled out when the probability lies below the threshold. Count the patients with and without disease among the ruled out and among the referred, and compute the sensitivity, the specificity and the negative predictive value, the share free of disease among the patients ruled out. Give each proportion with its 95 percent Wilson interval: For k of n, with z = 1.96, the centre is (k/n plus z squared over 2n) divided by (1 plus z squared over n), and the half-width is z times the square root of (k/n times one minus k/n, divided by n, plus z squared over 4n squared), divided by (1 plus z squared over n).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the table of the four counts, print the three proportions with their Wilson intervals, and keep the number of patients ruled out under the name "answer".  
> Test: The four counts must add up to 76, and every interval must lie between 0 and 1 and contain its proportion.  

In [ ]:
# Data of Prompt 3.2: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.2
teach.check_step("Prompt 3.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.2 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "disease": _c["y_test"].to_numpy(),
                     "probability, model": np.round(_c["p_model"], 6)})


def wilson(k, n, z=1.96):
    c = (k / n + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(k / n * (1 - k / n) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return round(float(c - h), 3), round(float(c + h), 3)


y = data["disease"].to_numpy()
out = data["probability, model"].to_numpy() < 0.18
TP, FN = int((~out & (y == 1)).sum()), int((out & (y == 1)).sum())
TN, FP = int((out & (y == 0)).sum()), int((~out & (y == 0)).sum())
print(pd.DataFrame({"disease": [TP, FN], "no disease": [FP, TN]}, index=["referred", "ruled out"]))
print("sensitivity", round(TP / (TP + FN), 3), wilson(TP, TP + FN))
print("specificity", round(TN / (TN + FP), 3), wilson(TN, TN + FP))
print("negative predictive value", round(TN / (TN + FN), 3), wilson(TN, TN + FN))
answer = int(out.sum())
teach.check_step("Prompt 3.2", "answer", globals())

**What the result shows.** The threshold rules out 27 of the 76 test patients without missing any patient with disease. The lower bound of the negative predictive value is still far from what a rule-out needs, which is why the next stage is a larger validation.

#### Prompt 3.3: Women and men

**Goal.** Give the sensitivity of the rule-out in women and in men with the lower bounds of their Wilson intervals.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A rule-out has to be safe in every group in which it is used. The test set holds fewer women than men, and fewer of the women have disease.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, "sex" is "man" or "woman", and "probability, model" the probability of coronary artery disease from a logistic regression of the course on eight clinical variables, fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise. The rule-out threshold of the project, chosen on the training patients, is 0.18.  
> Task: For women and for men separately, count the patients, the patients with disease, the patients with disease at or above the threshold and the patients ruled out, and compute the sensitivity. Give each proportion with its 95 percent Wilson interval: For k of n, with z = 1.96, the centre is (k/n plus z squared over 2n) divided by (1 plus z squared over n), and the half-width is z times the square root of (k/n times one minus k/n, divided by n, plus z squared over 4n squared), divided by (1 plus z squared over n).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each sex, with the lower bound of the Wilson interval of the sensitivity, and keep the lower bound for women, with three decimals, under the name "answer".  
> Test: The patients of the two rows must add up to 76, and a sensitivity of 1 must have an upper bound of 1.  

In [ ]:
# Data of Prompt 3.3: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "sex": _c["X_test"]["sex"].map({1: "man", 0: "woman"}).to_numpy(),
                     "disease": _c["y_test"].to_numpy(), "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.3
teach.check_step("Prompt 3.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.3 (run it only if the code of the assistant does not pass the check)
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "sex": _c["X_test"]["sex"].map({1: "man", 0: "woman"}).to_numpy(),
                     "disease": _c["y_test"].to_numpy(), "probability, model": np.round(_c["p_model"], 6)})


def wilson(k, n, z=1.96):
    c = (k / n + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    h = z * np.sqrt(k / n * (1 - k / n) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return round(float(c - h), 3), round(float(c + h), 3)


d = data
out = d["probability, model"].to_numpy() < 0.18
y = d["disease"].to_numpy()
rows = []
for group in ["woman", "man"]:
    g = (d["sex"] == group).to_numpy()
    k, n = int((~out & (y == 1) & g).sum()), int(((y == 1) & g).sum())
    rows.append({"group": group, "patients": int(g.sum()), "with disease": n,
                         "sensitivity": round(k / n, 3), "lower bound": wilson(k, n)[0],
                         "ruled out": int((out & g).sum())})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
answer = float(table.iloc[0]["lower bound"])
teach.check_step("Prompt 3.3", "answer", globals())

**What the result shows.** No woman with disease is missed, and still the lower bound of the sensitivity in women is only 0.646: Seven women with disease cannot show that the rule-out is safe for women. The report names this limit, and the plan of the validation asks for enough women.

**«Stop 3 · Prompt and application» is complete.** Next comes «Stop 3 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity».

### Stop 3 · Ready application

This part chooses a rule-out threshold on the training patients, reports what the model uses and where it may fail, writes the table of results of the report and the toolbox of the final project, with the starting points of the options.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 3 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 2 · Ready application» and «Stop 3 · Python basics» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# functions of earlier parts, kept unchanged in the course package
from vclinic.week11 import bootstrap_auc, interval, calibration, slope_interval, wilson

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week11 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import importlib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
ALL = list(X.columns)
def fit(columns):
    """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)
reference, model = fit(REFERENCE), fit(ALL)
p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
p_model = model.predict_proba(X_test[ALL])[:, 1]
yt = y_test.to_numpy()
boot = bootstrap_auc(yt, p_ref, p_model)

#### A rule-out threshold

*Lecture page: [A decision threshold: Ruling out](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#a-decision-threshold-ruling-out)*

Five-fold cross-validation gives every training patient the probability of a model that did not see that patient. On a grid from 0.01 to 0.50, the project takes the highest threshold at which at least 95 percent of the training patients with disease lie at or above it. The threshold is then applied once to the test patients.

**From the formulas to the code.** The cross-validated probabilities: `p_cv`. The threshold: `threshold`. The patients ruled out: `ruled_out`. TP, FN, TN and FP: `counts`.

**On the lecture page:** [Worked example: Lower bounds when no patient is missed](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#worked-example-lower-bounds-when-no-patient-is-missed). The cell below chooses the threshold and computes the same lower bounds on the test patients.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
ruled_out = p_model < threshold
counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
          "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}

print("threshold", threshold, "| cross-validated sensitivity", round(float((p_cv[y_train.to_numpy() == 1] >= threshold).mean()), 3))
print("cross-validated calibration: intercept and slope", np.round(calibration(y_train.to_numpy(), p_cv), 3))
print(counts)
for name, k, n in (("sensitivity", counts["TP"], counts["TP"] + counts["FN"]), ("specificity", counts["TN"], counts["TN"] + counts["FP"]),
                   ("negative predictive value", counts["TN"], counts["TN"] + counts["FN"])):
    print(f"{name}: {k}/{n} = {k / n:.3f}", np.round(wilson(k, n), 3))
print("ruled out", int(ruled_out.sum()), "of", len(yt))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#nbref-c171)*

**Python note.** `cross_val_predict(..., cv=folds, method="predict_proba")[:, 1]` fits the pipeline five times, each time on four fifths of the training patients, and returns for every patient the probability from the fit that left that patient out. The *generator expression* `max(t for t in GRID if ...)` runs through the grid and keeps the highest threshold that meets the condition. `p_model < threshold` gives an array of `True` and `False`, and `~ruled_out` turns it around; `&` combines two such arrays patient by patient, so `(ruled_out & (yt == 1)).sum()` counts the patients with disease below the threshold.

In [ ]:
ts = np.round(np.arange(0.01, 0.51, 0.01), 2)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, p, y, title in ((axes[0], p_cv, y_train.to_numpy(), "training, cross-validated"), (axes[1], p_model, yt, "test")):
    ax.plot(ts, [(p[y == 1] >= t).mean() for t in ts], color="#9B1B1B", label="sensitivity")
    ax.plot(ts, [(p < t).mean() for t in ts], color="#1F5F8B", label="share ruled out")
    ax.axhline(0.95, color="#8A8A8A", linestyle="--", linewidth=1)
    ax.axvline(threshold, color="#231A1A", linestyle=":", linewidth=1)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("threshold")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Exercise G.** Which threshold would a target sensitivity of 0.90 on the cross-validated training patients give?

In [ ]:
data = exercise_table("G", X_train=X_train, p_cv=p_cv, y_train=y_train)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = float(max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.90))
teach.check("Exercise G", answer)

#### What the model uses

*Lecture page: [What the model uses and where it may fail](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#what-the-model-uses-and-where-it-may-fail)*

The full model is a logistic regression on standardised columns, so its coefficients can be compared with each other, and the contribution of a column to the log-odds of one patient is its coefficient times the z-score of the patient: the closed form of Week 9. The section explains the test patient with the lowest probability, Patient 260 of the case of the week.

**From the formulas to the code.** The coefficients per standard deviation: `coefficients`. The contributions of one patient: `contributions`. The patient with the lowest probability: `low`.

**Start cell.** For the section «What the model uses», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «What the model uses»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week11 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X, y = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
ALL = list(X.columns)
def fit(columns):
    """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)
reference, model = fit(REFERENCE), fit(ALL)
p_model = model.predict_proba(X_test[ALL])[:, 1]

**On the lecture page:** [Worked example: Why Patient 260 has a probability of 0.032](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#worked-example-why-patient-260-has-a-probability-of-0-032). The cell below computes the contributions of the variables of the same patient.

In [ ]:
scaler, logistic = model.named_steps["standardscaler"], model.named_steps["logisticregression"]
coefficients = pd.Series(logistic.coef_[0], index=ALL)


def contributions(patient):
    """What each column adds to the log-odds of the patient: the coefficient times the z-score (the closed form of Week 9)."""
    z = (patient[ALL].to_numpy(dtype=float) - scaler.mean_) / scaler.scale_
    return pd.Series(logistic.coef_[0] * z, index=ALL)

print(coefficients.sort_values(key=abs, ascending=False).round(3))
low = X_test.iloc[[int(np.argmin(p_model))]]
c_low = contributions(low.iloc[0])
eta = logistic.intercept_[0] + c_low.sum()
print("patient", low.index[0], "| intercept", round(logistic.intercept_[0], 3), "| sum of the contributions", round(c_low.sum(), 3),
      "| log-odds", round(eta, 3), "| probability", round(1 / (1 + np.exp(-eta)), 3), round(float(p_model.min()), 3))
fig, ax = plt.subplots(figsize=(7.2, 3.6))
order_ = c_low.sort_values().index
ax.barh(order_, c_low[order_], color=["#1F5F8B" if v < 0 else "#9B1B1B" for v in c_low[order_]])
ax.axvline(0, color="#231A1A", linewidth=1)
ax.set_xlabel("contribution to the log-odds")
ax.set_title(f"Patient {low.index[0]}", fontsize=9)
plt.tight_layout()
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#nbref-c180)*

**Python note.** `model.named_steps["standardscaler"]` and `model.named_steps["logisticregression"]` reach the two steps of the pipeline by their names. `scaler.mean_` and `scaler.scale_` are the means and the standard deviations that the scaler learned from the training patients, so `z` is the z-score of the patient in every column. `X_test.iloc[[k]]` with double brackets keeps a table of one row, and `.iloc[0]` takes that row as a series. `sort_values(key=abs, ascending=False)` orders the coefficients by their size, whatever their sign.

**Exercise H.** What probability would the model give Patient 260 if she were a man, with every other value unchanged? Keep it with three decimals.

In [ ]:
data = exercise_table("H", ALL=ALL, logistic=logistic, low=low, scaler=scaler)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
man = low.copy()
man["sex"] = 1
answer = round(float(model.predict_proba(man[ALL])[:, 1][0]), 3)
teach.check("Exercise H", answer)

#### Where the model may fail

*Lecture page: [What the model uses and where it may fail](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#what-the-model-uses-and-where-it-may-fail)*

A rule-out must be safe in every group in which it is used. The section repeats the sensitivity and the patients ruled out for the women and the men of the test set, with the lower bound of the Wilson interval of the sensitivity.

**From the lecture to the code.** The groups: `groups`. The table of the groups: `subgroups`.

In [ ]:
groups = {"women": X_test["sex"].to_numpy() == 0, "men": X_test["sex"].to_numpy() == 1}
rows = []
for name, g in groups.items():
    k_tn, n_neg = int((ruled_out & (yt == 0) & g).sum()), int(((yt == 0) & g).sum())
    k_tp, n_pos = int((~ruled_out & (yt == 1) & g).sum()), int(((yt == 1) & g).sum())
    rows.append({"group": name, "patients": int(g.sum()), "with disease": n_pos,
                 "sensitivity": k_tp / n_pos, "lower bound": wilson(k_tp, n_pos)[0],
                 "ruled out": int((ruled_out & g).sum())})
subgroups = pd.DataFrame(rows)

print(subgroups.round(3).to_string(index=False))

**Python note.** `groups` is a dictionary from the name of a group to a mask of `True` and `False` over the test patients. The loop computes, for each group, the patients with disease above the threshold, `k_tp`, out of all patients with disease of the group, `n_pos`, and `wilson(k_tp, n_pos)[0]` takes the lower bound of the interval. The same loop works for any grouping: an age band, a hospital or a period.

**Observe and reflect.** No woman with disease is missed, and still the lower bound of the sensitivity in women is about 0.65. How many women with disease would a validation need to show a lower bound of 0.95, and how many women in all if a third of them had disease?

**Exercise I.** How many test patients aged 60 or more does the rule-out threshold spare an angiography?

In [ ]:
data = exercise_table("I", X_test=X_test, p_model=p_model, yt=yt)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((ruled_out & (X_test["age"].to_numpy() >= 60)).sum())
teach.check("Exercise I", answer)

#### The table of results of the report

*Lecture page: [Reporting with TRIPOD+AI](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#reporting-with-tripod-ai)*

TRIPOD+AI asks for every measure of performance with its uncertainty, overall and in the groups in which the model would be used. The section collects all results of the worked project in one table with their intervals, writes it into `final_project/results/results.csv` and draws it as a figure for the slides.

**From the lecture to the code.** The rows of the report: `results`. The table: `table`.

In [ ]:
project = pathlib.Path("final_project")
for name in ["results", "figures"]:
    (project / name).mkdir(parents=True, exist_ok=True)
auc_ref, auc_model = roc_auc_score(yt, p_ref), roc_auc_score(yt, p_model)
results = [("AUC of the reference", "all", auc_ref, *interval(boot["a"])),
           ("AUC of the model", "all", auc_model, *interval(boot["b"])),
           ("difference of the AUCs", "all", auc_model - auc_ref, *interval(boot["difference"])),
           ("calibration slope", "all", calibration(yt, p_model)[1], *slope_interval(yt, p_model))]
sex = X_test["sex"].to_numpy()
for group, g in (("all", np.ones(len(yt), dtype=bool)), ("women", sex == 0), ("men", sex == 1)):
    for name, k, n in (("sensitivity", int((~ruled_out & (yt == 1) & g).sum()), int(((yt == 1) & g).sum())),
                       ("specificity", int((ruled_out & (yt == 0) & g).sum()), int(((yt == 0) & g).sum())),
                       ("negative predictive value", int((ruled_out & (yt == 0) & g).sum()), int((ruled_out & g).sum()))):
        results.append((name, group, k / n, *wilson(k, n)))
table = pd.DataFrame(results, columns=["result", "group", "value", "lower bound", "upper bound"]).round(3)
table.to_csv(project / "results" / "results.csv", index=False)

print(table.to_string(index=False))
plot = table[table["result"] != "calibration slope"]
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ypos = np.arange(len(plot))[::-1]
ax.errorbar(plot["value"], ypos, xerr=[plot["value"] - plot["lower bound"], plot["upper bound"] - plot["value"]],
            fmt="o", color="#1F5F8B", ecolor="#8A8A8A", capsize=3)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{r}, {g}" for r, g in zip(plot["result"], plot["group"])], fontsize=8)
ax.axvline(0, color="#231A1A", linewidth=0.8)
ax.set_xlim(-0.1, 1.05)
ax.set_title("Results of the worked project with their 95 percent intervals (UCI Heart Disease, Cleveland, CC BY 4.0)", fontsize=9)
plt.tight_layout()
fig.savefig(project / "figures" / "results.png", dpi=150)
plt.show()

**Python note.** `*interval(boot["a"])` unpacks the two ends of the interval into the tuple, so that each row holds five values. The double loop runs through three groups and three proportions, and each row gets its Wilson interval; `pd.DataFrame(results, columns=[...])` names the five columns, and `to_csv(..., index=False)` writes the table without its row numbers. `plt.errorbar` draws each value with its interval, and `fig.savefig(...)` keeps the figure in the folder of the project for the slides.

**Exercise J.** What is the lower bound of the Wilson interval of the negative predictive value at the threshold of 0.26? Keep it with three decimals.

In [ ]:
data = exercise_table("J", X_test=X_test, p_model=p_model, yt=yt)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
out_26 = p_model < 0.26
answer = round(wilson(int((out_26 & (yt == 0)).sum()), int(out_26.sum()))[0], 3)
teach.check("Exercise J", answer)

#### The toolbox of the final project and the starting points of the options

The checks of this week do not depend on the Cleveland data. They need three arrays of the same length: the outcomes of the test patients, the probabilities of a simple reference and the probabilities of the model of the project. The cell below writes them into the module `final_project/code/project_tools.py`, with a docstring for every function, and the next cell runs the module once on the worked project; its numbers are those of the ready applications of Stops 2 and 3.

| Option | Start from | Outcome `y` | Reference `p_reference` | Model `p_model` |
|---|---|---|---|---|
| A. ECG | Notebooks of Weeks 1 and 5 | A rhythm or ischaemia label of the test ECGs | A rule of one measurement, for example the QRS duration or the ST elevation | The network of Week 5 |
| B. Risk | Notebooks of Weeks 2 and 9, and this notebook | Disease, or an event during follow-up | A model of the variables the clinic already uses | The model of the project |
| C. Imaging | Notebooks of Weeks 3, 4 and 7 | A low ejection fraction, or a diagnosis of the images | The measurement of the expert, or a simpler network | The network of the project |
| D. Text | Notebook of Week 6 | The finding in the synthetic report, as read by the truth of the virtual clinic | A search for key words | The extraction of the project; with `wilson` for its accuracy |
| E. Fusion | Notebook of Week 8 | The outcome of the registry of the virtual clinic | The best model of a single kind of data | The model that joins them |
| F. Decision support | Notebooks of Weeks 9 and 10 | The truth of the records of the user test | Version 1.0 of the prototype | The improved version |
| G. Own question | Any week | The outcome of the clinic, defined before the analysis | What the clinic does now, as a score or a rule | The model of the project, on the computer of the institution |

In [ ]:
pathlib.Path("final_project/code").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile final_project/code/project_tools.py
"""Toolbox of the final project: the checks of Week 11 for any model of a binary outcome.

Each function works with three arrays of the same length: y, the outcomes (0 or 1); p_reference,
the probabilities of a simple reference; p_model, the probabilities of the model of the project.
"""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def wilson(k, n, z=1.96):
    """The Wilson interval of k of n."""
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


def bootstrap_difference(y, p_reference, p_model, n_boot=2000, seed=2026):
    """The AUCs of the reference and of the model and their difference in resamples of the patients."""
    y, a, b = np.asarray(y), np.asarray(p_reference), np.asarray(p_model)
    rng, rows = np.random.default_rng(seed), []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() < y[i].max():
            ra, rb = roc_auc_score(y[i], a[i]), roc_auc_score(y[i], b[i])
            rows.append({"reference": ra, "model": rb, "difference": rb - ra})
    return pd.DataFrame(rows)


def percentile_interval(values, level=0.95):
    """The percentile interval of bootstrap values."""
    return tuple(np.percentile(values, [50 * (1 - level), 100 - 50 * (1 - level)]))


def calibration(y, p, steps=50):
    """Calibration intercept (with the log-odds as offset) and slope, by Newton steps."""
    y, lp = np.asarray(y, dtype=float), np.log(np.asarray(p) / (1 - np.asarray(p)))
    a = 0.0
    for _ in range(steps):
        q = 1 / (1 + np.exp(-(a + lp)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    Z, b = np.c_[np.ones_like(lp), lp], np.zeros(2)
    for _ in range(steps):
        q = 1 / (1 + np.exp(-Z @ b))
        b += np.linalg.solve(Z.T @ (Z * (q * (1 - q))[:, None]), Z.T @ (y - q))
    return float(a), float(b[1])


def rule_out_threshold(p_cv, y, target=0.95, grid=np.round(np.arange(0.01, 0.51, 0.01), 2)):
    """The highest threshold of the grid with a sensitivity of at least target, on cross-validated training probabilities."""
    p_cv, y = np.asarray(p_cv), np.asarray(y)
    return float(max(t for t in grid if (p_cv[y == 1] >= t).mean() >= target))


def rule_out_table(y, p, threshold, groups=None):
    """Sensitivity, specificity and negative predictive value at a threshold, with Wilson intervals, overall and in groups."""
    y, out = np.asarray(y), np.asarray(p) < threshold
    groups = {"all": np.ones(len(y), dtype=bool), **(groups or {})}
    rows = []
    for name, g in groups.items():
        for result, k, n in (("sensitivity", (~out & (y == 1) & g).sum(), ((y == 1) & g).sum()),
                             ("specificity", (out & (y == 0) & g).sum(), ((y == 0) & g).sum()),
                             ("negative predictive value", (out & (y == 0) & g).sum(), (out & g).sum())):
            if n > 0:
                rows.append({"result": result, "group": name, "k": int(k), "n": int(n), "value": k / n,
                             "lower bound": wilson(k, n)[0], "upper bound": wilson(k, n)[1]})
    return pd.DataFrame(rows)

In [ ]:
sys.path.insert(0, str(pathlib.Path("final_project/code").resolve()))
import project_tools as tools
importlib.reload(tools)
b2 = tools.bootstrap_difference(yt, p_ref, p_model)
print("difference of the AUCs", np.round(tools.percentile_interval(b2["difference"]), 3))
print("calibration", np.round(tools.calibration(yt, p_model), 3))
t2 = tools.rule_out_threshold(p_cv, y_train, target=0.95)
print("threshold", t2)
print(tools.rule_out_table(yt, p_model, t2, {"women": X_test["sex"].to_numpy() == 0, "men": X_test["sex"].to_numpy() == 1}).round(3).to_string(index=False))

**Python note.** A cell that starts with `%%writefile final_project/code/project_tools.py` is not run: Jupyter writes its text into the file, which then holds a small module. `sys.path.insert(0, ...)` adds the folder of the module to the places where Python looks for modules, `import project_tools as tools` reads it under the short name `tools`, and `importlib.reload(tools)` reads it again after you change the file. The functions take arrays and dictionaries of masks, so the same five lines evaluate the model of any option.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#designing-the-clinical-evaluation) and continue reading at «Designing the evaluation of clinical validity». The weekly project is at the end of this Jupyter notebook, and the section «Weekly project» of the lecture page describes it.

## Weekly project

*Lecture page: [Weekly project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#weekly-project) · [README of Week 11, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-11/README.md#weekly-project)*

**The project.** The final project of the course, with the steps of the worked project of this notebook: the question, the audit of the data and their size, a simple reference, every result with an interval, the calibration, a threshold chosen on the training data only, what the model uses and where it may fail, and a plan for the next stage of its clinical evaluation.

This part stands on its own: Its start cell loads the data of the project and prepares the tables of the steps. The project can be built with the prompts below, one for each step; each step works like a prompt of the stops, without a check cell or a ready code, because the results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

**Start cell.** For the weekly project, the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the weekly project
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=11, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

import math
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _course():
    """The worked project of the week: the Cleveland data, the split of Week 2, the reference, the full model and the cross-validated probabilities of the training patients (computed once, a few seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    heart = load_cleveland(source=DATA_SOURCE)
    FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
    missing = heart.isna().sum()
    audit = pd.DataFrame({"missing values": missing, "used by the project": [c in FEATURES for c in missing.index]})

    cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
    X, y = cad.drop(columns="disease"), cad["disease"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)


    def cox_snell_r2(y, p):
        """The Cox-Snell R squared of predicted probabilities p for the outcomes y."""
        y, p = np.asarray(y, dtype=float), np.clip(np.asarray(p, dtype=float), 1e-12, 1 - 1e-12)
        phi = y.mean()
        ll_null = len(y) * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))
        ll_model = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        return 1 - np.exp(2 * (ll_null - ll_model) / len(y))


    def riley_sizes(P, prevalence, r2_cs, shrinkage=0.9, delta=0.05, margin=0.05):
        """The three sizes of Riley and colleagues for a model of a binary outcome with P parameters: (i) a shrinkage of at least 0.9, (ii) apparent and adjusted Nagelkerke R squared within 0.05, (iii) the overall risk within 0.05."""
        r2_max = 1 - np.exp(2 * (prevalence * np.log(prevalence) + (1 - prevalence) * np.log(1 - prevalence)))
        n1 = P / ((shrinkage - 1) * np.log(1 - r2_cs / shrinkage))
        s2 = r2_cs / (r2_cs + delta * r2_max)
        n2 = P / ((s2 - 1) * np.log(1 - r2_cs / s2))
        n3 = (1.96 / margin) ** 2 * prevalence * (1 - prevalence)
        return {"(i)": math.ceil(n1), "(ii)": math.ceil(n2), "(iii)": math.ceil(n3), "R2_max": round(float(r2_max), 3)}

    REFERENCE = ["age", "sex", "cp_2.0", "cp_3.0", "cp_4.0"]
    anticipated = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[REFERENCE], y_train)
    r2_cs = cox_snell_r2(y_train, anticipated.predict_proba(X_train[REFERENCE])[:, 1])
    P = X_train.shape[1]
    sizes = riley_sizes(P, float(y_train.mean()), r2_cs)

    ALL = list(X.columns)


    def fit(columns):
        """A logistic regression on the training patients; the scaling is learned inside the pipeline, from the training patients only."""
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train[columns], y_train)


    reference, model = fit(REFERENCE), fit(ALL)
    p_ref = reference.predict_proba(X_test[REFERENCE])[:, 1]
    p_model = model.predict_proba(X_test[ALL])[:, 1]
    yt = y_test.to_numpy()

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    p_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_train[ALL], y_train, cv=folds, method="predict_proba")[:, 1]
    GRID = np.round(np.arange(0.01, 0.51, 0.01), 2)
    threshold = max(t for t in GRID if (p_cv[y_train.to_numpy() == 1] >= t).mean() >= 0.95)
    ruled_out = p_model < threshold
    counts = {"TP": int((~ruled_out & (yt == 1)).sum()), "FN": int((ruled_out & (yt == 1)).sum()),
              "TN": int((ruled_out & (yt == 0)).sum()), "FP": int((~ruled_out & (yt == 0)).sum())}
    p_ref_train = reference.predict_proba(X_train[REFERENCE])[:, 1]
    globals()["_course_cache"] = dict(heart=heart, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, p_ref=p_ref,
                                      p_model=p_model, p_cv=p_cv, p_ref_train=p_ref_train, threshold=threshold)
    return _course_cache


def _patients(which):
    """The training or the test patients, or both in the order of the split, with their set, the eight clinical variables and the disease."""
    c = _course()
    parts = {"training": c["X_train"].index, "test": c["X_test"].index}
    rows = [(s, i) for s, idx in parts.items() for i in idx if which == "both" or s == which]
    h = c["heart"].loc[[i for _, i in rows]]
    return pd.DataFrame({"patient": h.index, "set": [s for s, _ in rows],
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "chest pain type": h["cp"].map(_CP),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "disease": h["disease"]}).reset_index(drop=True)

### Project step 1: The audit of the data and the split

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A cardiologist builds a final project on ruling out coronary artery disease after an exercise test. Before any model, the data are audited and split once into training and test patients. For a final project of your own, rewrite the Data line for your own table: its columns, their units and the meaning of the outcome, in words, and never a row of a patient.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test, with all their columns. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302). The eight variables of the project are "age" (years), "sex" ("man" or "woman"), "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina) and "ST depression" (the depression of the ST segment during exercise compared with rest, mm). Five more columns come from the data set: "fasting blood sugar above 120" (1 when the fasting blood sugar was above 120 mg/dL), "resting ECG" (0 normal, 1 an abnormality of the ST segment or the T wave, 2 left ventricular hypertrophy), "ST slope" (the slope of the ST segment at peak exercise: 1 up, 2 flat, 3 down), "vessels on fluoroscopy" (the number of major vessels coloured by fluoroscopy, 0 to 3) and "thallium scan" (3 normal, 6 fixed defect, 7 reversible defect). An empty cell is a missing value. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Give the number of patients, the number and the share with disease, and the missing values of every column. Then split the patients once, with all rows of a patient in the same set: a quarter of them for the test, with the same share of disease in both sets, using the seed 2026 for the random numbers. Give the numbers of patients and of patients with disease in each set.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the table of the missing values and the table of the two sets, and keep the number of test patients with disease under the name "answer".  
> Test: The two sets must add up to all patients, and no patient may be in both.  

In [ ]:
# Data of project step 1: the table "data" that the prompt describes
_h = _course()["heart"]
data = pd.DataFrame({"patient": _h.index, "age": _h["age"], "sex": _h["sex"].map({1.0: "man", 0.0: "woman"}),
                     "chest pain type": _h["cp"].map(_CP), "resting blood pressure": _h["trestbps"],
                     "cholesterol": _h["chol"], "fasting blood sugar above 120": _h["fbs"],
                     "resting ECG": _h["restecg"], "maximum heart rate": _h["thalach"],
                     "exercise angina": _h["exang"], "ST depression": _h["oldpeak"], "ST slope": _h["slope"],
                     "vessels on fluoroscopy": _h["ca"], "thallium scan": _h["thal"],
                     "disease": _h["disease"]}).reset_index(drop=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: A reference, the model and the interval of the gain

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist compares the full model with what is known before the exercise test, on the same test patients, and gives the gain with its interval.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina) and "ST depression" (the depression of the ST segment during exercise compared with rest, mm). "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Make the inputs of the reference: the age, a column that is 1 for men and 0 for women, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, so that typical angina is the reference category. Turn each input into a z-score with the mean and the standard deviation of the training patients, and fit the usual logistic regression for disease, with its standard settings and at most 1,000 iterations, on the training patients only. The full model uses the same inputs and adds the resting blood pressure, the cholesterol, the maximum heart rate, exercise angina and the ST depression, fitted in the same way. Compute the probability of disease of both models for every test patient and their AUCs on the test patients. Draw 2,000 resamples of the test patients: Each resample draws as many patients as the table has, with replacement, with a random number generator started with the seed 2026. Skip a resample that holds patients of one outcome only. In each resample, compute the area under the ROC curve (AUC) of the reference and of the model on the same patients, and the difference, model minus reference. The 95 percent interval of a result runs from the 2.5th to the 97.5th percentile of its values over the resamples.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with the AUCs of the two models and of the difference with their intervals, and keep the 2.5th percentile of the difference, with three decimals, under the name "answer".  
> Test: No test patient may be used to fit either model, and both models must be judged on the same resamples.  

In [ ]:
# Data of project step 2: the table "data" that the prompt describes
data = _patients("both")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: Calibration and the decision threshold

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist chooses the rule-out threshold of the full model on the training patients and checks the calibration before the test patients are opened.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina) and "ST depression" (the depression of the ST segment during exercise compared with rest, mm). "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise.  
> Task: Fit the full model of step 2 by five-fold cross-validation on the training patients, with the folds drawn with the same share of disease and the seed 2026, so that every training patient gets the probability of a model that did not see that patient. On these probabilities, compute the calibration intercept and slope as in Stop 2, and choose the highest threshold among 0.01, 0.02 and so on up to 0.50 at which at least 95 percent of the training patients with disease have a probability at or above it. Then fit the full model on all training patients, apply the threshold once to the test patients and count the patients ruled out, with and without disease.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the calibration of the cross-validated probabilities, the threshold and the counts on the test patients, and keep the number of test patients ruled out under the name "answer".  
> Test: The threshold must be chosen before any test patient is used, and the sensitivity on the training patients at the threshold must be at least 0.95.  

In [ ]:
# Data of project step 3: the table "data" that the prompt describes
data = _patients("both")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: The table of results for the report

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The report of the final project follows TRIPOD+AI: Every result comes with its interval, overall and in the groups in which the model would be used.  
> Data: The notebook holds a table named "data" with the 76 test patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography after an exercise test. Each row is a patient: "patient" gives the number of the patient in the data, "sex" is "man" or "woman", "probability, reference" the probability of coronary artery disease from a reference model of the course (a logistic regression on the age, the sex and the chest pain type), and "probability, model" the probability from the full model of the course (the same with the resting blood pressure, the cholesterol and three results of the exercise test added). Both models were fitted on the 227 training patients only. "disease" is 1 when the angiogram showed more than 50 percent narrowing of the diameter of at least one major coronary artery, and 0 otherwise. The rule-out threshold of the project, chosen on the training patients, is 0.18.  
> Task: Draw 2,000 resamples of the test patients: Each resample draws as many patients as the table has, with replacement, with a random number generator started with the seed 2026. Skip a resample that holds patients of one outcome only. In each resample, compute the area under the ROC curve (AUC) of the reference and of the model on the same patients, and the difference, model minus reference. The 95 percent interval of a result runs from the 2.5th to the 97.5th percentile of its values over the resamples. Then, at the rule-out threshold of 0.18, compute the sensitivity, the specificity and the negative predictive value for all test patients, for women and for men. Give each proportion with its 95 percent Wilson interval: For k of n, with z = 1.96, the centre is (k/n plus z squared over 2n) divided by (1 plus z squared over n), and the half-width is z times the square root of (k/n times one minus k/n, divided by n, plus z squared over 4n squared), divided by (1 plus z squared over n). Write all results into one table with the columns result, group, value, lower bound and upper bound, ready to be copied into the report.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the table, and keep the lower bound of the negative predictive value of all test patients, with three decimals, under the name "answer".  
> Test: Every interval must contain its value and lie between 0 and 1, except the interval of the difference of the AUCs, which can be negative.  

In [ ]:
# Data of project step 4: the table "data" that the prompt describes
_c = _course()
data = pd.DataFrame({"patient": _c["X_test"].index, "sex": _c["X_test"]["sex"].map({1: "man", 0: "woman"}).to_numpy(),
                     "disease": _c["y_test"].to_numpy(), "probability, reference": np.round(_c["p_ref"], 6),
                     "probability, model": np.round(_c["p_model"], 6)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

**End of the weekly project.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-11/lecture.html#weekly-project): The section «Weekly project» describes the report and its submission.

## Application challenges by field

| Field | Challenge |
|---|---|
| Coronary artery disease and imaging | Replace the outcome of the worked project by your own definition of disease, for example a narrowing of 70 percent or an ischaemia on imaging, if your data have it, and run the ready applications again, from the section «A reference and the model» to the section «Where the model may fail». Report how the threshold and the patients ruled out change. |
| Heart failure | For a model of an event during follow-up, write down the time horizon of the outcome before the analysis, and use `tools.rule_out_table` for the patients that the model would not call back early. |
| Electrophysiology | Take the AF model of Week 10 as Option F: version 1.0 as the reference and version 1.1 as the model on the same test minutes, and give the difference of their AUCs with its bootstrap interval. |
| Echocardiography | For a network that flags a low ejection fraction, use the measurement of the expert as the reference, and report the sensitivity in the patients with poor image quality as a group of its own. |
| Emergency department | A rule-out at the emergency department sends patients home. Plan the size of a validation for a negative predictive value whose lower bound is 0.99, and compare it with the patients that your department sees in a year. |
| Quality and patient safety | Write the rule of monitoring for your model after its introduction: which calibration slope, measured every quarter on which patients, would make the clinic stop and review it. |
| Research and ethics | Write the paragraph of the application to an ethics committee for the silent run of your model: the data, where they stay, who sees the output, why the patients are not affected, and the plan of the analysis with its size. |

## After the notebook

This notebook took one final project from the audit of its data to the table of its report and a toolbox that works for every option. The final project of each participant follows the same steps with a question of their own clinic, and the session closes with the presentations and the certificates. The materials of all eleven weeks stay open: Each week can be followed again with its study path.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>